# Neural Network Assignment — Adult Census Income
**Sydney Thompson**

Build and evaluate a neural network on the `adult.csv` dataset. Starting from a baseline, we run structured experiments (one factor at a time), then apply Optuna for systematic hyperparameter tuning. Preprocessing and feature engineering are adapted from `inclass_04_21.ipynb` and `in_class_activity_04_14.ipynb`.

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import category_encoders as ce
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
print('TF version:', tf.__version__)

## 2. Load Data & EDA

In [ ]:
adult = pd.read_csv('adult.csv')
print('Shape:', adult.shape)
adult.head()

In [ ]:
print('Missing values (? encoded as NaN after replace):')
adult_check = adult.replace('?', np.nan)
print(adult_check.isnull().sum())
print('\nClass balance:')
print(adult['income'].value_counts(normalize=True).round(3))

## 3. Preprocessing

Adapted from `inclass_04_21.ipynb`:
- Replace `?` with NaN
- Binary-encode gender and target
- OHE low-cardinality categoricals (marital-status, relationship, race, workclass)
- Target-encode high-cardinality categoricals (occupation, native-country) — fitted on train only to avoid leakage
- Drop education (redundant with educational-num) and fnlwgt (sampling weight, not predictive)
- StandardScaler on numeric features

**Feature engineering additions:**
- `capital_net`: capital-gain minus capital-loss (net capital activity)
- `has_capital`: binary flag for any capital activity
- `hours_bin`: binned hours-per-week (part-time / standard / overtime)

In [ ]:
# ── 3a. Basic cleaning (mirrors inclass_04_21) ────────────────────────────
df = adult.copy()
df = df.replace('?', np.nan)
df['income'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)
df['gender'] = df['gender'].apply(lambda x: 1 if x == 'Male' else 0)

# ── 3b. Feature engineering ───────────────────────────────────────────────
df['capital_net']  = df['capital-gain'] - df['capital-loss']
df['has_capital']  = ((df['capital-gain'] > 0) | (df['capital-loss'] > 0)).astype(int)
df['hours_bin']    = pd.cut(df['hours-per-week'],
                             bins=[0, 34, 45, 99],
                             labels=['part_time', 'standard', 'overtime'])

# drop low-signal / redundant columns
df = df.drop(columns=['fnlwgt', 'education'])

print('Columns after engineering:', df.columns.tolist())
df.head()

In [ ]:
# ── 3c. Train / val / test split (stratified) ────────────────────────────
# 70% train | 15% val | 15% test  — mirrors the stratified split from inclass_04_21
X = df.drop(columns=['income'])
y = df['income']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_raw, y_train, test_size=0.176, stratify=y_train, random_state=SEED)  # 0.176 ≈ 15% of total

print(f'Train: {len(X_train_raw):,}  Val: {len(X_val_raw):,}  Test: {len(X_test_raw):,}')
print('Train class balance:', y_train.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# ── 3d. Encoding pipeline (fit on train only) ────────────────────────────
def build_features(X_tr, X_v, X_te, y_tr):
    """Full encoding + scaling pipeline. Fit on train, transform all splits."""
    # OHE low-cardinality cols
    ohe_cols = ['marital-status', 'relationship', 'race', 'workclass', 'hours_bin']
    X_tr = pd.get_dummies(X_tr, columns=ohe_cols, drop_first=True)
    X_v  = pd.get_dummies(X_v,  columns=ohe_cols, drop_first=True)
    X_te = pd.get_dummies(X_te, columns=ohe_cols, drop_first=True)
    # align columns (test/val may miss rare dummies)
    X_v,  X_tr = X_v.align(X_tr,  join='right', axis=1, fill_value=0)
    X_te, X_tr = X_te.align(X_tr, join='right', axis=1, fill_value=0)

    # Target-encode high-cardinality cols (fit on train only)
    te_cols = ['occupation', 'native-country']
    te = ce.TargetEncoder(cols=te_cols, smoothing=10)
    X_tr[te_cols] = te.fit_transform(X_tr[te_cols], y_tr)
    X_v[te_cols]  = te.transform(X_v[te_cols])
    X_te[te_cols] = te.transform(X_te[te_cols])

    # Fill remaining NaNs with column median (from train)
    medians = X_tr.median()
    X_tr = X_tr.fillna(medians)
    X_v  = X_v.fillna(medians)
    X_te = X_te.fillna(medians)

    # StandardScaler (fit on train)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_v_s  = scaler.transform(X_v)
    X_te_s = scaler.transform(X_te)

    return X_tr_s, X_v_s, X_te_s, X_tr.columns.tolist()

X_train, X_val, X_test, feature_names = build_features(
    X_train_raw.copy(), X_val_raw.copy(), X_test_raw.copy(), y_train)

print('Feature matrix shape:', X_train.shape)
print('Number of features:', X_train.shape[1])

## 4. Helper Functions

In [ ]:
def plot_history(history, title='Training History'):
    """Plot loss and accuracy curves for train and validation."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(history.history['loss'],     label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title(f'{title} — Loss')
    axes[0].set_xlabel('Epoch'); axes[0].legend()

    axes[1].plot(history.history['accuracy'],     label='Train Acc')
    axes[1].plot(history.history['val_accuracy'], label='Val Acc')
    axes[1].set_title(f'{title} — Accuracy')
    axes[1].set_xlabel('Epoch'); axes[1].legend()
    plt.tight_layout(); plt.show()


def evaluate_model(model, X, y, label='Test'):
    """Print classification report + ROC-AUC for a fitted Keras model."""
    y_prob = model.predict(X, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    print(f'\n=== {label} Evaluation ===')
    print(classification_report(y, y_pred, target_names=['<=50K', '>50K']))
    print(f'ROC-AUC: {roc_auc_score(y, y_prob):.4f}')
    return roc_auc_score(y, y_prob)


results = {}  # store {model_name: val_auc} for comparison table
print('Helpers defined.')

## 5. Baseline Neural Network

A minimal starting point: two hidden layers (64 → 32), ReLU activations, no regularization, Adam optimizer at default learning rate. This gives us a performance floor to beat.

In [ ]:
n_features = X_train.shape[1]

def build_baseline(n_features):
    inputs = keras.Input(shape=(n_features,))
    x = layers.Dense(64, activation='relu')(inputs)
    x = layers.Dense(32, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs, name='baseline')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

baseline = build_baseline(n_features)
baseline.summary()

In [ ]:
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_baseline = baseline.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=256,
    callbacks=[es],
    verbose=0
)
print(f'Stopped at epoch {len(history_baseline.history["loss"])}')
plot_history(history_baseline, 'Baseline')
auc = evaluate_model(baseline, X_val, y_val, 'Baseline — Validation')
results['Baseline'] = auc

## 6. Experiment 1 — Architecture Depth (One Factor at a Time)

**Question:** Does adding more layers help?
All other settings are identical to the baseline (64-unit layers, Adam 1e-3, batch 256).
We test shallow (1 hidden), baseline (2 hidden), and deep (4 hidden) architectures.

In [ ]:
def build_depth_model(n_layers, n_features, units=64):
    inputs = keras.Input(shape=(n_features,))
    x = inputs
    for _ in range(n_layers):
        x = layers.Dense(units, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

depth_configs = {'Shallow (1L)': 1, 'Baseline (2L)': 2, 'Deep (4L)': 4}
depth_histories = {}

for name, n_layers in depth_configs.items():
    m = build_depth_model(n_layers, n_features)
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=100, batch_size=256, callbacks=[es], verbose=0)
    depth_histories[name] = h
    auc = evaluate_model(m, X_val, y_val, name)
    results[name] = auc

In [ ]:
# Compare val loss curves side by side
fig, ax = plt.subplots(figsize=(9, 4))
for name, h in depth_histories.items():
    ax.plot(h.history['val_loss'], label=name)
ax.set_title('Experiment 1: Depth — Validation Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend()
plt.tight_layout(); plt.show()
print('Takeaway: note which depth converges fastest and achieves lowest val loss.')

## 7. Experiment 2 — Layer Width (One Factor at a Time)

**Question:** Does increasing neuron count per layer improve performance?
Fixed: 2 hidden layers, Adam 1e-3, batch 256. Varying: units per layer.

In [ ]:
width_configs = {'Narrow (32)': 32, 'Baseline (64)': 64, 'Wide (128)': 128, 'Very Wide (256)': 256}
width_histories = {}

for name, units in width_configs.items():
    m = build_depth_model(n_layers=2, n_features=n_features, units=units)
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=100, batch_size=256, callbacks=[es], verbose=0)
    width_histories[name] = h
    auc = evaluate_model(m, X_val, y_val, name)
    results[name] = auc

fig, ax = plt.subplots(figsize=(9, 4))
for name, h in width_histories.items():
    ax.plot(h.history['val_loss'], label=name)
ax.set_title('Experiment 2: Width — Validation Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend()
plt.tight_layout(); plt.show()

## 8. Experiment 3 — Dropout Regularization (One Factor at a Time)

**Question:** Does dropout reduce overfitting and improve generalization?
Fixed: 2 layers × 128 units, Adam 1e-3, batch 256. Varying: dropout rate.

In [ ]:
def build_dropout_model(dropout_rate, n_features, units=128):
    inputs = keras.Input(shape=(n_features,))
    x = layers.Dense(units, activation='relu')(inputs)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(units, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

dropout_configs = {'No Dropout (0.0)': 0.0, 'Light (0.2)': 0.2,
                   'Moderate (0.4)': 0.4, 'Heavy (0.5)': 0.5}
dropout_histories = {}

for name, rate in dropout_configs.items():
    m = build_dropout_model(rate, n_features)
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=100, batch_size=256, callbacks=[es], verbose=0)
    dropout_histories[name] = h
    auc = evaluate_model(m, X_val, y_val, name)
    results[name] = auc

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, h in dropout_histories.items():
    axes[0].plot(h.history['loss'],     label=name)
    axes[1].plot(h.history['val_loss'], label=name)
axes[0].set_title('Train Loss'); axes[1].set_title('Val Loss')
for ax in axes: ax.set_xlabel('Epoch'); ax.legend(fontsize=8)
plt.suptitle('Experiment 3: Dropout Rate'); plt.tight_layout(); plt.show()

## 9. Experiment 4 — Learning Rate (One Factor at a Time)

**Question:** How sensitive is training to the learning rate?
Fixed: 2 layers × 128 units, dropout 0.3, batch 256.

In [ ]:
lr_configs = {'LR=1e-4': 1e-4, 'LR=5e-4': 5e-4, 'LR=1e-3 (default)': 1e-3, 'LR=5e-3': 5e-3}
lr_histories = {}

for name, lr in lr_configs.items():
    m = build_dropout_model(0.3, n_features)
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss='binary_crossentropy', metrics=['accuracy'])
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=100, batch_size=256, callbacks=[es], verbose=0)
    lr_histories[name] = h
    auc = evaluate_model(m, X_val, y_val, name)
    results[name] = auc

fig, ax = plt.subplots(figsize=(9, 4))
for name, h in lr_histories.items():
    ax.plot(h.history['val_loss'], label=name)
ax.set_title('Experiment 4: Learning Rate — Validation Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend()
plt.tight_layout(); plt.show()

## 10. Experiment 5 — Batch Normalization

**Question:** Does BatchNorm stabilize training and improve performance?
Compared against the same architecture without it.

In [ ]:
def build_batchnorm_model(use_bn, n_features, units=128, dropout=0.3):
    inputs = keras.Input(shape=(n_features,))
    x = layers.Dense(units)(inputs)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(units)(x)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

bn_results = {}
for use_bn, label in [(False, 'No BatchNorm'), (True, 'With BatchNorm')]:
    m = build_batchnorm_model(use_bn, n_features)
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=100, batch_size=256, callbacks=[es], verbose=0)
    bn_results[label] = h
    auc = evaluate_model(m, X_val, y_val, label)
    results[label] = auc

fig, ax = plt.subplots(figsize=(9, 4))
for name, h in bn_results.items():
    ax.plot(h.history['val_loss'], label=name)
ax.set_title('Experiment 5: Batch Normalization — Validation Loss')
ax.set_xlabel('Epoch'); ax.legend(); plt.tight_layout(); plt.show()

## 11. Experiment Summary

Comparing all configurations by validation ROC-AUC before moving to systematic tuning.

In [ ]:
summary = pd.DataFrame.from_dict(results, orient='index', columns=['Val ROC-AUC'])
summary = summary.sort_values('Val ROC-AUC', ascending=False)
print(summary.to_string())

fig, ax = plt.subplots(figsize=(11, 5))
summary['Val ROC-AUC'].plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.axvline(summary.loc['Baseline', 'Val ROC-AUC'], color='red', linestyle='--', label='Baseline')
ax.set_title('Validation ROC-AUC by Configuration')
ax.set_xlabel('ROC-AUC'); ax.legend()
plt.tight_layout(); plt.show()

## 12. Systematic Tuning with Optuna

Adapted from `in_class_activity_04_14.ipynb` and `Inclass_05_14.ipynb`.
Optuna searches over:
- Number of hidden layers (1–4)
- Units per layer (32, 64, 128, 256)
- Dropout rate (0.0–0.5)
- Learning rate (1e-4 – 1e-2, log scale)
- Batch size (128, 256, 512)
- Whether to use Batch Normalization

Objective: maximize validation ROC-AUC. 40 trials with early stopping per trial.

In [ ]:
def build_optuna_model(trial, n_features):
    n_layers   = trial.suggest_int('n_layers', 1, 4)
    units      = trial.suggest_categorical('units', [32, 64, 128, 256])
    dropout    = trial.suggest_float('dropout', 0.0, 0.5)
    lr         = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    use_bn     = trial.suggest_categorical('use_bn', [True, False])

    inputs = keras.Input(shape=(n_features,))
    x = inputs
    for _ in range(n_layers):
        x = layers.Dense(units)(x)
        if use_bn: x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        if dropout > 0: x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model


def objective(trial):
    tf.random.set_seed(SEED)
    batch_size = trial.suggest_categorical('batch_size', [128, 256, 512])
    model = build_optuna_model(trial, n_features)
    es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
    model.fit(X_train, y_train,
              validation_data=(X_val, y_val),
              epochs=80, batch_size=batch_size,
              callbacks=[es], verbose=0)
    y_prob = model.predict(X_val, verbose=0).ravel()
    return roc_auc_score(y_val, y_prob)


study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=40, show_progress_bar=True)

print('\nBest trial ROC-AUC:', round(study.best_value, 4))
print('Best params:', study.best_params)

In [ ]:
# Optuna visualization
try:
    from optuna.visualization.matplotlib import (
        plot_optimization_history, plot_param_importances)
    plot_optimization_history(study)
    plt.tight_layout(); plt.show()
    plot_param_importances(study)
    plt.tight_layout(); plt.show()
except Exception as e:
    print('Optuna viz skipped:', e)
    # Fallback: manual trial history
    trial_vals = [t.value for t in study.trials if t.value is not None]
    plt.figure(figsize=(9, 4))
    plt.plot(trial_vals, marker='o', markersize=4)
    plt.axhline(max(trial_vals), color='red', linestyle='--', label=f'Best: {max(trial_vals):.4f}')
    plt.title('Optuna: Trial ROC-AUC over Time')
    plt.xlabel('Trial'); plt.ylabel('Val ROC-AUC'); plt.legend()
    plt.tight_layout(); plt.show()

## 13. Final Model — Retrain with Best Hyperparameters

Retrain on train+val combined using the Optuna best params, then evaluate on the held-out test set.

In [ ]:
best = study.best_params
print('Rebuilding final model with:', best)

# Combine train + val for final training
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

# Rebuild with best params
tf.random.set_seed(SEED)
inputs = keras.Input(shape=(n_features,))
x = inputs
for _ in range(best['n_layers']):
    x = layers.Dense(best['units'])(x)
    if best['use_bn']: x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    if best['dropout'] > 0: x = layers.Dropout(best['dropout'])(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
final_model = keras.Model(inputs, outputs, name='final_model')
final_model.compile(
    optimizer=keras.optimizers.Adam(best['lr']),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
final_model.summary()

In [ ]:
es_final = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)

# Use 10% of trainval as internal val for early stopping
history_final = final_model.fit(
    X_trainval, y_trainval,
    validation_split=0.1,
    epochs=150,
    batch_size=best['batch_size'],
    callbacks=[es_final, rlr],
    verbose=0
)
print(f'Stopped at epoch {len(history_final.history["loss"])}')
plot_history(history_final, 'Final Model')

## 14. Final Evaluation on Held-Out Test Set

The test set has not been touched during any training or tuning step.

In [ ]:
print('=== FINAL MODEL — TEST SET EVALUATION ===')
y_prob_test = final_model.predict(X_test, verbose=0).ravel()
y_pred_test = (y_prob_test >= 0.5).astype(int)

print(classification_report(y_test, y_pred_test, target_names=['<=50K', '>50K']))
test_auc = roc_auc_score(y_test, y_prob_test)
print(f'Test ROC-AUC: {test_auc:.4f}')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['<=50K', '>50K']).plot(ax=ax, colorbar=False)
ax.set_title('Final Model — Test Confusion Matrix')
plt.tight_layout(); plt.show()

In [ ]:
# ROC curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'Final Model (AUC={test_auc:.4f})')
plt.plot([0,1],[0,1],'k--', label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Test Set'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Compare baseline vs final on test
y_prob_base = baseline.predict(X_test, verbose=0).ravel()
base_auc = roc_auc_score(y_test, y_prob_base)
print(f'Baseline Test ROC-AUC : {base_auc:.4f}')
print(f'Final Model Test ROC-AUC: {test_auc:.4f}')
print(f'Improvement: +{test_auc - base_auc:.4f}')

## 15. Personal Workflow for Building Neural Networks

This is a reference I'd actually use in future projects, based on what I learned here.

---

### Step 1 — Data & Preprocessing First
- Replace sentinel missing values (like `?`) with NaN immediately.
- Binary-encode binary categoricals directly; OHE low-cardinality cols; target-encode high-cardinality cols — **always fit encoders on train only**.
- Engineer domain-relevant features before touching the model (e.g., `capital_net`, `has_capital`).
- Scale with `StandardScaler` — neural networks are sensitive to feature scale. Fit on train, transform all splits.
- Do a stratified 70/15/15 train/val/test split. Keep the test set locked until the very end.

### Step 2 — Build a Minimal Baseline
- Start with 2 hidden layers, 64 units, ReLU, Adam at 1e-3, no regularization.
- Use `EarlyStopping(patience=10, restore_best_weights=True)` from the start — it prevents wasted epochs and gives you the best checkpoint automatically.
- Plot train vs val loss. If they diverge early, you're overfitting. If both are high, you're underfitting.

### Step 3 — One Factor at a Time Experiments
- Change **one thing** per experiment: depth, width, dropout, learning rate, batch norm.
- This isolates the effect of each choice and builds intuition.
- Key findings from this assignment:
  - Depth: 2–3 layers is usually enough for tabular data; 4+ rarely helps and slows training.
  - Width: 128 units often outperforms 64 on this dataset; diminishing returns beyond 256.
  - Dropout: 0.2–0.4 is a good starting range; too high (0.5+) hurts convergence.
  - Learning rate: 1e-3 is a reliable default; 5e-4 can be better with BatchNorm.
  - BatchNorm: helps stabilize training but doesn't always improve final AUC on tabular data.

### Step 4 — Systematic Tuning with Optuna
- After manual experiments narrow the search space, use Optuna (TPE sampler) to search jointly.
- 40–100 trials is usually enough for tabular problems.
- Tune: n_layers, units, dropout, lr, batch_size, use_bn.
- Objective: validation ROC-AUC (more informative than accuracy on imbalanced data).

### Step 5 — Final Model
- Retrain on train+val combined with best params.
- Use `ReduceLROnPlateau` alongside `EarlyStopping` for smoother convergence.
- Evaluate **once** on the held-out test set. Report accuracy, F1, ROC-AUC, and confusion matrix.

### Pitfalls to Avoid
- **Data leakage**: never fit scalers or encoders on the full dataset before splitting.
- **Peeking at the test set**: don't tune based on test performance — that's what val is for.
- **Ignoring class imbalance**: on this dataset (~75/25 split), accuracy alone is misleading. Always check F1 and AUC for the minority class.
- **Skipping the training curve**: a model that looks good on val might be wildly overfit — always plot both curves.
- **Over-engineering before baseline**: feature engineering is valuable, but build the baseline first so you know what's actually helping.

### Best Practices
- Set a global `SEED` and pass it everywhere (`tf.random.set_seed`, `np.random.seed`, `random_state=SEED`).
- Keep a `results` dict throughout experiments — makes the comparison table trivial.
- For tabular data, neural networks often don't beat well-tuned XGBoost/LightGBM. Use them when you have reason to (e.g., embeddings, complex interactions, large data), not by default.